In [27]:
# =====================================================
# Anime Recommendation System using Cosine Similarity
# =====================================================

# Step 1: Import Libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [28]:
# Step 2: Load Dataset
df = pd.read_csv("anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [29]:
# Step 3: Display Dataset Information
print("First 5 Rows:")
print(df.head())

First 5 Rows:
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  


In [30]:
print("\nDataset Shape:")
print(df.shape)


Dataset Shape:
(12294, 7)


In [31]:
print("\nColumn Names:")
print(df.columns)


Column Names:
Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')


In [32]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [33]:
# Step 4: Handle Missing Values

# Fill Genre with empty string
df['genre'] = df['genre'].fillna('')

In [34]:
# Fill rating with median
df['rating'] = df['rating'].fillna(df['rating'].median())

In [35]:
# Fill episodes if required
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

# Fill members
df['members'] = df['members'].fillna(df['members'].median())

In [36]:
# Step 5: Convert Genre into TF-IDF Features

tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(df['genre'])

In [37]:
# Step 6: Normalize Numerical Features

scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(
    df[['rating', 'episodes', 'members']]
)

In [38]:
# Step 7: Combine Genre + Numerical Features

from scipy.sparse import hstack

combined_features = hstack([genre_matrix, numerical_features])

In [39]:
# Step 8: Calculate Cosine Similarity

cosine_sim = cosine_similarity(combined_features)

print("\nCosine Similarity Matrix Shape:")
print(cosine_sim.shape)



Cosine Similarity Matrix Shape:
(12294, 12294)


In [40]:
# Step 9: Recommendation Function

def recommend_anime(title, similarity_matrix=cosine_sim, top_n=10):
    
    if title not in df['name'].values:
        return "Anime not found."
    
    index = df[df['name'] == title].index[0]
    
    similarity_scores = list(enumerate(similarity_matrix[index]))
    
    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)
    
    similarity_scores = similarity_scores[1:top_n+1]
    
    anime_indices = [i[0] for i in similarity_scores]
    
    recommendations = df[['name', 'genre', 'rating']].iloc[anime_indices]
    
    return recommendations

In [41]:
# Step 10: Test Recommendation System

anime_name = "Naruto"

print("\nRecommended Anime:\n")

print(recommend_anime(anime_name))


Recommended Anime:

                                                   name  \
615                                  Naruto: Shippuuden   
206                                       Dragon Ball Z   
346                                         Dragon Ball   
1472        Naruto: Shippuuden Movie 4 - The Lost Tower   
1573  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
486                            Boruto: Naruto the Movie   
1343                                        Naruto x UT   
2997  Naruto Soyokazeden Movie: Naruto to Mashin to ...   
588                                     Dragon Ball Kai   
1103  Boruto: Naruto the Movie - Naruto ga Hokage ni...   

                                                  genre  rating  
615   Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
206   Action, Adventure, Comedy, Fantasy, Martial Ar...    8.32  
346   Adventure, Comedy, Fantasy, Martial Arts, Shou...    8.16  
1472  Action, Comedy, Martial Arts, Shounen, Super P...    7.53  

In [42]:
# Step 11: Recommendation with Threshold

def recommend_with_threshold(title, threshold=0.40):
    
    if title not in df['name'].values:
        return "Anime not found."
    
    idx = df[df['name']==title].index[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = [(i,score) for i,score in sim_scores if score >= threshold]
    
    sim_scores = sorted(sim_scores,
                        key=lambda x:x[1],
                        reverse=True)
    
    sim_scores = sim_scores[1:]
    
    indices = [i[0] for i in sim_scores]
    
    return df[['name','genre','rating']].iloc[indices]

print("\nRecommendation using Threshold:\n")
print(recommend_with_threshold("Naruto", threshold=0.50))


Recommendation using Threshold:

                                                    name  \
615                                   Naruto: Shippuuden   
206                                        Dragon Ball Z   
346                                          Dragon Ball   
1472         Naruto: Shippuuden Movie 4 - The Lost Tower   
1573   Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
...                                                  ...   
58                           Kuroko no Basket 3rd Season   
591                             Kami nomi zo Shiru Sekai   
1083                                       Inu x Boku SS   
4434                              Battle Spirits: Heroes   
11834                                        Mitama: Nin   

                                                   genre  rating  
615    Action, Comedy, Martial Arts, Shounen, Super P...    7.94  
206    Action, Adventure, Comedy, Fantasy, Martial Ar...    8.32  
346    Adventure, Comedy, Fantasy, Martial A

In [ ]:
1. User-based vs. item-based collaborative filtering

User-based CF finds users who behave similarly to you (rate or watch the same anime you do) and recommends items those similar users liked that you haven't seen yet. It relies on building a user-user similarity matrix.
Item-based CF instead builds an item-item similarity matrix — it finds items similar to ones you ve already rated highly, based on how other users collectively rated them (not the item's own content/attributes, but shared rating patterns). It then recommends items most similar to what you already like.

Key practical difference: item-based CF tends to scale better and stay more stable over time, because item-item relationships (e.g., "people who liked A also liked B") change more slowly than an individual user's taste profile — and the user base itself keeps growing and shifting. That's why item-based CF became the industry default at large scale (Amazon's original recommender system is a classic example).

2. What is collaborative filtering, and how does it work?

Collaborative filtering is a recommendation technique that predicts what a user will like based on the past behavior (ratings, clicks, purchases) of many users, rather than analyzing the content of the items themselves. Its core assumption: if people agreed in the past, they'll likely agree again in the future.

It works by building a user-item interaction matrix (rows = users, columns = items, values = ratings/interactions), then using one of two main approaches:

Memory-based approach: directly computes similarity between users or items from that matrix (as described in Q1) and recommends based on nearest neighbors.
Model-based approach: learns hidden ("latent") factors from the matrix — for example, using matrix factorization or SVD — that represent underlying traits like genre preference or pacing, and uses those learned factors to predict ratings for items a user hasn't rated yet.